In [ ]:
import numpy as np
import scipy.linalg as slin
import scipy.optimize as sopt
from scipy.special import expit as sigmoid
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import utils
from sklearn.model_selection import KFold
utils.set_random_seed(8)

In [18]:
def notears_linear(X, lambda1, loss_type, w_threshold, max_iter=100, h_tol=1e-8, rho_max=1e+16):
    """Solve min_W L(W; X) + lambda1 ‖W‖_1 s.t. h(W) = 0 using augmented Lagrangian.
    Args:
        X (np.ndarray): [n, d] sample matrix
        lambda1 (float): l1 penalty parameter
        loss_type (str): l2, logistic, poisson
        max_iter (int): max num of dual ascent steps
        h_tol (float): exit if |h(w_est)| <= htol
        rho_max (float): exit if rho >= rho_max
        w_threshold (float): drop edge if |weight| < threshold

    Returns:
        W_est (np.ndarray): [d, d] estimated DAG
    """
    def _loss(W):
        """Evaluate value and gradient of loss."""
        M = X @ W
        if loss_type == 'l2':
            R = X - M
            loss = 0.5 / X.shape[0] * (R ** 2).sum()
            G_loss = - 1.0 / X.shape[0] * X.T @ R
        elif loss_type == 'logistic':
            loss = 1.0 / X.shape[0] * (np.logaddexp(0, M) - X * M).sum()
            G_loss = 1.0 / X.shape[0] * X.T @ (sigmoid(M) - X)
        elif loss_type == 'poisson':
            S = np.exp(M)
            loss = 1.0 / X.shape[0] * (S - X * M).sum()
            G_loss = 1.0 / X.shape[0] * X.T @ (S - X)
        else:
            raise ValueError('unknown loss type')
        return loss, G_loss

    def _h(W):
        """Evaluate value and gradient of acyclicity constraint."""
        E = slin.expm(W * W)  # (Zheng et al. 2018)
        h = np.trace(E) - d
        #     # A different formulation, slightly faster at the cost of numerical stability
        #     M = np.eye(d) + W * W / d  # (Yu et al. 2019)
        #     E = np.linalg.matrix_power(M, d - 1)
        #     h = (E.T * M).sum() - d
        G_h = E.T * W * 2
        return h, G_h

    def _adj(w):
        """Convert doubled variables ([2 d^2] array) back to original variables ([d, d] matrix)."""
        return (w[:d * d] - w[d * d:]).reshape([d, d])

    def _func(w):
        """Evaluate value and gradient of augmented Lagrangian for doubled variables ([2 d^2] array)."""
        W = _adj(w)
        loss, G_loss = _loss(W)
        h, G_h = _h(W)
        obj = loss + 0.5 * rho * h * h + alpha * h + lambda1 * w.sum()
        G_smooth = G_loss + (rho * h + alpha) * G_h
        g_obj = np.concatenate((G_smooth + lambda1, - G_smooth + lambda1), axis=None)
        return obj, g_obj

    n, d = X.shape
    w_est, rho, alpha, h = np.zeros(2 * d * d), 1.0, 0.0, np.inf  # double w_est into (w_pos, w_neg)
    bnds = [(0, 0) if i == j else (0, None) for _ in range(2) for i in range(d) for j in range(d)]
    if loss_type == 'l2':
        X = X - np.mean(X, axis=0, keepdims=True)
    for _ in range(max_iter):
        w_new, h_new = None, None
        while rho < rho_max:
            sol = sopt.minimize(_func, w_est, method='L-BFGS-B', jac=True, bounds=bnds)
            w_new = sol.x
            h_new, _ = _h(_adj(w_new))
            if h_new > 0.25 * h:
                rho *= 10
            else:
                break
        w_est, h = w_new, h_new
        alpha += rho * h
        if h <= h_tol or rho >= rho_max:
            break
    W_est = _adj(w_est)
    W_est[np.abs(W_est) < w_threshold] = 0
    return W_est


In [ ]:
# DAG und so erzeugen. 
knoten = 20
kanten = 20
n, d, s0, graph_type, sem_type = 100, knoten, kanten, 'ER', 'gauss'
B_true = utils.simulate_dag(d, s0, graph_type)
W_true = utils.simulate_parameter(B_true)
X = utils.simulate_linear_sem(W_true, n, sem_type)


### Cross-Validation to tune the cutoff-parameter

Problem: One hyperparameter of the NoTears Algorithm is the cutoff parameter, wich determines the minimum value a weight in the estimated DAG needs to have in order to be included into the model. The use of the hyperparameter is to force acyclity as well as remove edges with small weights wich are less likely to be related to real underlying causal relationships. 

In the experiments, the authors sampled weights from the intervals {[], []} and set the cutoff parameter for all experiments on randomly generated Bayesian Nets to 0.3. In real data this may be a problem: If the causal relationships in the data are important but weak, they are likely to not be modelled, since the estimates are cut off at the threshold of 0.3. Aware of that problem, the authors suggested a more data-driven choice of the cutoff parameter, wich we tried to achieve using different experimental setups.

An obvious method that could be used to tune the hyperparameter is cross validation: Since one big advantage of the method is it's fast computation of the DAG, and CV is a generic and model-agnostic way to tune hyperparameters, this seemed like a good choice. 

The thing left to do is choosing an appropriate evaluation metric, wich resulted in the first set back: A metric suggested by the authors is the mean squared error of reconstruction the data points.

    For a data-point x from our test data, we calculate the **ToDo**

The problem arising from this metric, is the following: The algorithm trains a sparse DAG, since a penalty term using L1 regularization is added. If we remove even more edges from the dag using the cutoff parameter, the reconstruction error only goes up. So the CV always suggests no cutoff for the final model, even if the underlying ground truth known from generating the data implies that a higher cutoff would give better results. 

Our idea was to "fine-tune" this squared reconstruction error, by adding a penalty term aswell. Naturally a good idea was to penalize more edges, or vice-versa to reward sparser DAGs that still reconstruct the test data well. Thus we added a penalty term adding the number of estimated non-zero edges scaled by a small parameter of 0.01. In order to be able to find this scalar parameter independent of the input, eg. the number of nodes (and unknown number of edges), we normalized this penalty term by dividing the number of non-zero edges by the number of nodes in our network. 

Thus we added a penalty of 'estimated edges per node' to our reconstruction score. 


To-Do: Test for Different Numbers of
            ->  Data Points
            ->  Nodes
            ->  Edges (Especially Nodes to Edges-Ratios)
            ->  Noise Levels
            ->  Maybe Regularisierung durch Fisher Information? 

In [20]:
# Threshold über CV Estimaten, und damit Vanilla Algorithmus laufen lassen

thresholds = np.arange(0, 0.35, 0.01)
kf = KFold(n_splits=5, shuffle=True, random_state=4)
mse_values = []
shd_values = []
tpr_values = []
fpr_values = []

# Maybe -> Linkes Ende des Test-Intervalls Minimum Threshold, so dass DAG
# Cutoff im Original Paper -> Auf 0.3 gesetzt, mit Parametern in der Simulation >0.5, ist klar dass das dann gut klappt.
# Was wenn die Parameter kleiner werden (siehe Problem Tilman/Bootstrap) -> Cutoff muss schlauer gewählt werden!

for threshold in thresholds:
        fold_mses = []
        for train_idx, test_idx in kf.split(X):
            X_train, X_test = X[train_idx], X[test_idx]
            W_est = notears_linear(X_train, lambda1=0.1, loss_type='l2', w_threshold=threshold)
            X_pred = X_test @ W_est
            nonzero = np.count_nonzero(W_est)/knoten    #Nenner ist hier Anzahl der Knoten im DAG -> nonzero also Anzahl der Kanten ungleich Null pro Knoten
            lambdo = 0.01
            mse = np.mean((X_test - X_pred) ** 2) + lambdo * nonzero
            fold_mses.append(mse)
        
        W_est_full = notears_linear(X, lambda1=0.1, loss_type='l2', w_threshold=threshold)
        acc = utils.count_accuracy(B_true, W_est_full != 0)
        shd_values.append(acc['shd'])
        tpr_values.append(acc['tpr'])
        fpr_values.append(acc['fpr'])
        avg_mse = np.mean(fold_mses)
        mse_values.append(avg_mse)
        print(f"Threshold: {threshold:.2f}, MSE: {avg_mse:.8f}, SHD: {acc['shd']}, TPR: {acc['tpr']:.4f}, FPR: {acc['fpr']:.4f}")

# Find threshold with minimal MSE
min_mse = np.min(mse_values)
min_mse_idx = np.argmin(mse_values)
best_threshold = thresholds[min_mse_idx]

# Find threshold with minimal SHD
min_shd = np.min(shd_values)
min_shd_idx = np.argmin(shd_values)
best_shd_threshold = thresholds[min_shd_idx]

# SHD for threshold 0.3 (if in thresholds)
if 0.3 in thresholds:
    idx_03 = np.where(thresholds == 0.3)[0][0]
    shd_03 = shd_values[idx_03]
else:
    shd_03 = None
    
shd_cv = shd_values[min_mse_idx]

print(f"\nCross-Validated Threshold bei {best_threshold:.4f} mit SHD: {shd_cv} ")

print(f"Beste SHD bei Threshold: {best_shd_threshold:.4f} mit SHD von {min_shd}")

if shd_03 is not None:
    print(f"SHD für Threshold aus dem Paper (0.3): {shd_03}")
else:
    print("Threshold 0.3 not in tested thresholds.")



# Estimate full model using the best threshold
W_est_best = notears_linear(X, lambda1=0.1, loss_type='l2', w_threshold=best_threshold)

Threshold: 0.00, MSE: 1.07652404, SHD: 70, TPR: 1.0000, FPR: 0.6444
Threshold: 0.01, MSE: 1.03774780, SHD: 46, TPR: 0.7000, FPR: 0.2556
Threshold: 0.02, MSE: 1.03554242, SHD: 41, TPR: 0.7000, FPR: 0.2278
Threshold: 0.03, MSE: 1.03359170, SHD: 31, TPR: 0.7000, FPR: 0.1722
Threshold: 0.04, MSE: 1.03088580, SHD: 26, TPR: 0.7000, FPR: 0.1444
Threshold: 0.05, MSE: 1.02771802, SHD: 22, TPR: 0.6000, FPR: 0.1167
Threshold: 0.06, MSE: 1.02499150, SHD: 19, TPR: 0.6000, FPR: 0.1000
Threshold: 0.07, MSE: 1.02113048, SHD: 16, TPR: 0.6000, FPR: 0.0833
Threshold: 0.08, MSE: 1.02126809, SHD: 15, TPR: 0.6000, FPR: 0.0778
Threshold: 0.09, MSE: 1.02109376, SHD: 12, TPR: 0.6000, FPR: 0.0611
Threshold: 0.10, MSE: 1.01765248, SHD: 12, TPR: 0.6000, FPR: 0.0556
Threshold: 0.11, MSE: 1.01699501, SHD: 10, TPR: 0.5000, FPR: 0.0389
Threshold: 0.12, MSE: 1.01014114, SHD: 9, TPR: 0.5000, FPR: 0.0333
Threshold: 0.13, MSE: 1.00994156, SHD: 7, TPR: 0.5000, FPR: 0.0222
Threshold: 0.14, MSE: 1.00913318, SHD: 7, TPR: 0.5